In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
from scipy.optimize import nnls
import zipfile
import os

# Define the precise hex colors requested by the user
color_crude = '#FFADAD'  # Red/Light Pink
color_recon = '#A0C4FF'  # Blue
color_iso = '#BDB2FF'    # Purple (Major component)
color_bor = '#FFC6FF'    # Pink (Minor component)
line_style_recon = '-'   # Solid line, NO dashes

# Load data
df_crude = pd.read_csv("Crude Reaction Mixture.csv", header=None, names=['ppm', 'intensity'])
df_borneol = pd.read_csv("Bomeol _minor_.csv", header=None, names=['ppm', 'intensity'])
df_iso = pd.read_csv("Isoborneol.csv", header=None, names=['ppm', 'intensity'])

# Clean and sort data
def prepare_df(df):
    return df.groupby('ppm').mean().reset_index().sort_values('ppm')

df_crude = prepare_df(df_crude)
df_borneol = prepare_df(df_borneol)
df_iso = prepare_df(df_iso)

# Create common ppm axis for mathematical combination
min_ppm = max(df_crude['ppm'].min(), df_borneol['ppm'].min(), df_iso['ppm'].min())
max_ppm = min(df_crude['ppm'].max(), df_borneol['ppm'].max(), df_iso['ppm'].max())
common_ppm = np.linspace(min_ppm, max_ppm, 3000)

def extract_and_normalize(df, common_x):
    f = interp1d(df['ppm'], df['intensity'], kind='linear', bounds_error=False, fill_value=0)
    y = f(common_x)
    baseline = np.percentile(y, 5) # Subtract ambient baseline noise
    y_norm = y - baseline
    y_norm[y_norm < 0] = 0
    return y_norm

y_iso_norm = extract_and_normalize(df_iso, common_ppm)
y_bor_norm = extract_and_normalize(df_borneol, common_ppm)
y_crude_norm = extract_and_normalize(df_crude, common_ppm)

# Use Non-Negative Least Squares (NNLS) to find exact multipliers
A = np.vstack([y_iso_norm, y_bor_norm]).T
coeffs, residual = nnls(A, y_crude_norm)
a_iso, a_bor = coeffs

# Mathematically reconstruct the crude mixture
y_reconstructed = (a_iso * y_iso_norm) + (a_bor * y_bor_norm)
y_residual = y_crude_norm - y_reconstructed

svg_files = []

# --- 1. STACKED SPECTRA ---
plt.figure(figsize=(12, 5))
plt.plot(df_iso['ppm'], df_iso['intensity'], color=color_iso, linewidth=2)
plt.plot(df_borneol['ppm'], df_borneol['intensity'], color=color_bor, linewidth=2)
plt.plot(df_crude['ppm'], df_crude['intensity'], color=color_crude, linewidth=2)
plt.gca().invert_xaxis()
plt.tick_params(axis='both', which='major', labelsize=12)
plt.tight_layout()
filename = '1_stacked_spectra.svg'
plt.savefig(filename, format='svg', bbox_inches='tight')
svg_files.append(filename)
plt.close()

# --- 2. INDIVIDUAL SPECTRA ---
fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
axes[0].plot(df_iso['ppm'], df_iso['intensity'] - np.percentile(df_iso['intensity'], 5), color=color_iso, linewidth=2)
axes[1].plot(df_borneol['ppm'], df_borneol['intensity'] - np.percentile(df_borneol['intensity'], 5), color=color_bor, linewidth=2)
axes[2].plot(df_crude['ppm'], df_crude['intensity'] - np.percentile(df_crude['intensity'], 5), color=color_crude, linewidth=2)
axes[2].invert_xaxis()
for ax in axes:
    ax.tick_params(axis='both', which='major', labelsize=10)
plt.tight_layout()
filename = '2_individual_spectra.svg'
plt.savefig(filename, format='svg', bbox_inches='tight')
svg_files.append(filename)
plt.close()



# --- 3A. CRUDE VS RECONSTRUCTED (NO INSETS, HIGHLIGHTER STYLE) ---
fig = plt.figure(figsize=(12, 7))
gs = fig.add_gridspec(2, 1, height_ratios=[3, 1], hspace=0.1)
ax_main = fig.add_subplot(gs[0])
ax_res = fig.add_subplot(gs[1], sharex=ax_main)

ax_main.plot(common_ppm, y_crude_norm, color=color_crude, linewidth=8, alpha=0.4, label='Crude')
ax_main.plot(common_ppm, y_reconstructed, color=color_recon, linestyle=line_style_recon, linewidth=1.5, label='Reconstructed')
ax_main.tick_params(axis='both', which='major', labelsize=12)
ax_res.plot(common_ppm, y_residual, color='gray', linewidth=1.5)
ax_res.axhline(0, color='black', linestyle='--', linewidth=1, alpha=0.5)
ax_res.tick_params(axis='both', which='major', labelsize=12)

ax_main.invert_xaxis()
plt.setp(ax_main.get_xticklabels(), visible=False)
filename = '3a_crude_vs_reconstructed_no_insets.svg'
plt.savefig(filename, format='svg', bbox_inches='tight')
svg_files.append(filename)
plt.close()



# --- 3A. CRUDE VS RECONSTRUCTED (WITH INSETS, HIGHLIGHTER STYLE) ---
fig = plt.figure(figsize=(12, 7))
gs = fig.add_gridspec(2, 1, height_ratios=[3, 1], hspace=0.1)
ax_main = fig.add_subplot(gs[0])
ax_res = fig.add_subplot(gs[1], sharex=ax_main)

ax_main.plot(common_ppm, y_crude_norm, color=color_crude, linewidth=8, alpha=0.4)
ax_main.plot(common_ppm, y_reconstructed, color=color_recon, linestyle=line_style_recon, linewidth=1.5)
ax_main.tick_params(axis='both', which='major', labelsize=12)
ax_res.plot(common_ppm, y_residual, color='gray', linewidth=1.5)
ax_res.axhline(0, color='black', linestyle='--', linewidth=1, alpha=0.5)
ax_res.tick_params(axis='both', which='major', labelsize=12)
ax_main.invert_xaxis()

axins1 = ax_main.inset_axes([0.15, 0.4, 0.25, 0.5])
axins1.plot(common_ppm, y_crude_norm, color=color_crude, linewidth=6, alpha=0.4)
axins1.plot(common_ppm, y_reconstructed, color=color_recon, linestyle=line_style_recon, linewidth=1.5)
axins1.set_xlim(4.05, 3.95)
y_zoom1 = y_crude_norm[(common_ppm <= 4.05) & (common_ppm >= 3.95)]
if len(y_zoom1) > 0: axins1.set_ylim(0, max(y_zoom1) * 1.2)
axins1.set_xticks([]); axins1.set_yticks([])
ax_main.indicate_inset_zoom(axins1, edgecolor="black", alpha=0.3)

axins2 = ax_main.inset_axes([0.45, 0.4, 0.25, 0.5])
axins2.plot(common_ppm, y_crude_norm, color=color_crude, linewidth=6, alpha=0.4)
axins2.plot(common_ppm, y_reconstructed, color=color_recon, linestyle=line_style_recon, linewidth=1.5)
axins2.set_xlim(3.65, 3.55)
y_zoom2 = y_crude_norm[(common_ppm <= 3.65) & (common_ppm >= 3.55)]
if len(y_zoom2) > 0: axins2.set_ylim(0, max(y_zoom2) * 1.2)
axins2.set_xticks([]); axins2.set_yticks([])
ax_main.indicate_inset_zoom(axins2, edgecolor="black", alpha=0.3)

plt.setp(ax_main.get_xticklabels(), visible=False)
filename = '3a_crude_vs_reconstructed_with_insets.svg'
plt.savefig(filename, format='svg', bbox_inches='tight')
svg_files.append(filename)
plt.close()

# --- 3B. SCALED COMPONENTS VS RECONSTRUCTED ---
plt.figure(figsize=(12, 5))
plt.fill_between(common_ppm, 0, a_iso * y_iso_norm, color=color_iso, alpha=0.7)
plt.fill_between(common_ppm, 0, a_bor * y_bor_norm, color=color_bor, alpha=0.7)
plt.plot(common_ppm, y_reconstructed, color=color_recon, linestyle=line_style_recon, linewidth=1.5)
plt.gca().invert_xaxis()
plt.tick_params(axis='both', which='major', labelsize=12)
plt.tight_layout()
filename = '3b_components_vs_reconstructed.svg'
plt.savefig(filename, format='svg', bbox_inches='tight')
svg_files.append(filename)
plt.close()

# Zip them all together for easy download
zip_filename = 'NMR_Spectra_SVGs.zip'
with zipfile.ZipFile(zip_filename, 'w') as zipf:
    for file in svg_files:
        zipf.write(file)

print(f"Successfully created {zip_filename} containing all SVG files.")